# Feature Circuits Phase C-1: Causal ATE per Edge

Phase B showed correlation up to 0.71 between MoE features across layers. Are those edges *causal*?

**Test**: for each top edge `f_i (L_A) -> f_j (L_B)`:
1. Forward baseline, measure `pre_j` at L_B
2. Forward with `f_i` ablated at L_A (subtract `z_i * W_dec_A[i]` from MoE output)
3. `ATE = mean(pre_j_baseline) - mean(pre_j_ablated)` at tokens where `f_i` fired

If `ATE > 0.1` and scales with `corr`, causality is confirmed. Circuit is real.

**Budget**: ~20-30 min. 5 prompts × ~14 forwards per prompt × batch 1.

**Output**: `phase_c/ate_summary.json` — ATE per edge + corr comparison.

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer')
    pip('uninstall', '-y', 'transformers', 'causal-conv1d')
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    pip('install','--no-cache-dir','flash-linear-attention')
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from safetensors import safe_open
from huggingface_hub import snapshot_download, HfApi
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/fc_phase_c'); OUT.mkdir(exist_ok=True)
HF_OUT = 'caiovicentino1/qwen36-feature-circuits'
HF_STAGE_B = 'caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b'
MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'

CAPTURE_LAYERS = [11, 17, 23]
D_MODEL = 2048
N_FEATURES = 4096
TOPK = 32
MAX_PROMPT_LEN = 1536
N_PROMPTS = 5

print('setup done')

## Cell 2 — Load model + Phase B MoE SAEs from HF

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print(f'Model: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Load MoE SAEs from Phase B
class TopKSAE(nn.Module):
    def __init__(self, d_in, n, k):
        super().__init__()
        self.W_enc = nn.Parameter(torch.zeros(d_in, n))
        self.b_enc = nn.Parameter(torch.zeros(n))
        self.W_dec = nn.Parameter(torch.zeros(n, d_in))
        self.b_dec = nn.Parameter(torch.zeros(d_in))
        self.k = k
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        top_v, top_i = pre.topk(self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, top_i, F.relu(top_v))
        return z, top_i, top_v

pB = snapshot_download(HF_OUT, repo_type='model', cache_dir='/content/cache',
                       allow_patterns=['phase_b/sae_moe_*.pt'])
saes_moe = {}
for L in CAPTURE_LAYERS:
    sae = TopKSAE(D_MODEL, N_FEATURES, TOPK)
    sd = torch.load(str(Path(pB)/f'phase_b/sae_moe_L{L}_n{N_FEATURES}_k{TOPK}.pt'),
                    map_location='cpu', weights_only=True)
    sae.load_state_dict(sd)
    for p in sae.parameters(): p.requires_grad_(False)
    saes_moe[L] = sae.cuda()
    print(f'L{L} SAE loaded')

def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

moe_modules = {}
for L in CAPTURE_LAYERS:
    layer = get_layer_module(model, L)
    moe_modules[L] = layer.mlp
print('MoE modules located')

## Cell 3 — Top edges (from Phase B Cell 8 output) + unique sources

In [ ]:
# Hardcoded from Phase B run (Cell 8 top-10 per pair)
top_edges = {
    (11, 17): [
        (752, 1137), (4007, 794), (3371, 794), (769, 1137),
        (4007, 3087), (3298, 1137), (4007, 1109), (4007, 2264),
        (4007, 1676), (769, 4003),
    ],
    (17, 23): [
        (3582, 1271), (3582, 694), (1149, 1541), (2469, 1541),
        (1137, 1271), (3582, 600), (2130, 1541), (3582, 3345),
        (3582, 3053), (1084, 2536),
    ],
    (11, 23): [
        (752, 694), (752, 1271), (752, 600), (3791, 2536),
        (2509, 2536), (769, 694), (752, 3053), (769, 1271),
        (769, 600), (3298, 1271),
    ],
}

# Unique source features per source layer
sources_by_layer = {11: set(), 17: set()}
for (a, b), edges in top_edges.items():
    for src, _ in edges:
        sources_by_layer[a].add(src)

sources_by_layer = {L: sorted(s) for L, s in sources_by_layer.items()}
for L, srcs in sources_by_layer.items():
    print(f'L{L} unique sources ({len(srcs)}): {srcs}')

## Cell 4 — Intervention + capture hooks

Global `intervene` dict controls whether to ablate at L_A. Capture hooks store MoE outputs at each layer.

In [ ]:
# State controlling intervention and capture
intervene = {'layer': None, 'feat': None}  # set before forward
captured_moe = {L: None for L in CAPTURE_LAYERS}

def make_intervene_capture_hook(L):
    def hook(module, inp, out):
        x = out[0] if isinstance(out, tuple) else out  # [B, T, d]
        # Intervene if this is the target layer
        if intervene['layer'] == L and intervene['feat'] is not None:
            sae_A = saes_moe[L]
            feat = intervene['feat']
            # Encode -> get z[..., feat] (post-topk)
            with torch.no_grad():
                pre = (x - sae_A.b_dec) @ sae_A.W_enc + sae_A.b_enc
                top_v, top_i = pre.topk(TOPK, dim=-1)
                z = torch.zeros_like(pre)
                z.scatter_(-1, top_i, F.relu(top_v))
                z_i = z[..., feat:feat+1]  # [B, T, 1]
                # Subtract feature i's decoder contribution
                delta = z_i * sae_A.W_dec[feat:feat+1, :]  # [B, T, d]
            x_new = x - delta
            captured_moe[L] = x_new.detach()
            if isinstance(out, tuple):
                return (x_new,) + out[1:]
            return x_new
        # No intervention: just capture
        captured_moe[L] = x.detach()
        return out
    return hook

handles = []
for L in CAPTURE_LAYERS:
    h = moe_modules[L].register_forward_hook(make_intervene_capture_hook(L))
    handles.append(h)
print(f'OK {len(handles)} intervene+capture hooks installed')

## Cell 5 — Load prompts

In [ ]:
corpus = snapshot_download(HF_STAGE_B, repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

questions = []
seen = set()
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q = meta['question']
        if q in seen: continue
        seen.add(q)
        opts = json.loads(meta['options'])
        if len(opts) != 10: continue
        questions.append(dict(question=q, options=opts))

random.seed(42); random.shuffle(questions)
questions = questions[:N_PROMPTS]
print(f'{len(questions)} prompts')

def format_prompt(q, opts):
    choices = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(opts))
    content = f"Answer the following multiple-choice question step by step.\n\nQuestion: {q}\n\nOptions:\n{choices}\n\nAnswer:"
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)

prompt_ids = []
for q in questions:
    p = format_prompt(q['question'], q['options'])
    ids = tok(p, return_tensors='pt').input_ids
    if ids.shape[1] <= MAX_PROMPT_LEN:
        prompt_ids.append(ids.cuda())
print(f'{len(prompt_ids)} usable prompts')

## Cell 6 — Baseline + intervention forwards, compute ATE

For each prompt:
1. Baseline forward → capture L_A and L_B MoE outputs
2. For each source f_i at L_A: intervene forward → capture L_B MoE
3. Compute pre_j baseline vs intervened at tokens where f_i fired at baseline
4. ATE(f_i → f_j) = mean(pre_j_baseline - pre_j_intervened) over firing tokens

In [ ]:
def compute_pre(sae, x):
    """x: [T, d]  ->  pre: [T, n_features]"""
    return ((x - sae.b_dec) @ sae.W_enc + sae.b_enc)

def compute_z(sae, x):
    pre = compute_pre(sae, x)
    top_v, top_i = pre.topk(TOPK, dim=-1)
    z = torch.zeros_like(pre)
    z.scatter_(-1, top_i, F.relu(top_v))
    return z

# Accumulators: ATE[(src_layer, tgt_layer)][src_feat] = running mean of pre_j delta [n_features]
ate_acc = {}  # (a, b, src_feat) -> [n_features] running mean
fire_counts = {}  # (a, b, src_feat) -> int (# firing tokens across prompts)

layer_pairs = [(11, 17), (11, 23), (17, 23)]

for prompt_idx, ids in enumerate(tqdm(prompt_ids, desc='prompts')):
    # ==== Baseline ====
    intervene['layer'] = None; intervene['feat'] = None
    with torch.no_grad():
        _ = model(input_ids=ids, attention_mask=torch.ones_like(ids))
    baseline_moe = {L: captured_moe[L][0].clone() for L in CAPTURE_LAYERS}
    baseline_pre = {L: compute_pre(saes_moe[L], baseline_moe[L]) for L in CAPTURE_LAYERS}
    baseline_z = {L: compute_z(saes_moe[L], baseline_moe[L]) for L in CAPTURE_LAYERS}

    # ==== Intervention per source feature per source layer ====
    for src_layer in [11, 17]:
        for src_feat in sources_by_layer[src_layer]:
            fire_mask = baseline_z[src_layer][:, src_feat] > 0  # [T]
            n_fire = int(fire_mask.sum().item())
            if n_fire == 0: continue  # nothing to measure

            intervene['layer'] = src_layer; intervene['feat'] = src_feat
            with torch.no_grad():
                _ = model(input_ids=ids, attention_mask=torch.ones_like(ids))
            # Downstream L_B (any layer > src_layer in capture list)
            for tgt_layer in [L for L in CAPTURE_LAYERS if L > src_layer]:
                intv_pre = compute_pre(saes_moe[tgt_layer], captured_moe[tgt_layer][0])
                # Per-token delta at firing tokens only
                delta = (baseline_pre[tgt_layer] - intv_pre)[fire_mask]  # [n_fire, n_features]
                delta_mean = delta.mean(0).float().cpu()  # [n_features]
                key = (src_layer, tgt_layer, src_feat)
                if key not in ate_acc:
                    ate_acc[key] = delta_mean * n_fire
                    fire_counts[key] = n_fire
                else:
                    ate_acc[key] = ate_acc[key] + delta_mean * n_fire
                    fire_counts[key] += n_fire

    intervene['layer'] = None; intervene['feat'] = None
    torch.cuda.empty_cache()

# Finalize: convert to mean
ate_final = {k: (v / fire_counts[k]).numpy() for k, v in ate_acc.items()}
print(f'\nComputed ATE for {len(ate_final)} (src_layer, tgt_layer, src_feat) combos')
for h in handles: h.remove()

## Cell 7 — Report ATE for top-20 edges + corr vs ATE table

In [ ]:
# Phase B correlations (from Cell 8 output, hardcoded for comparison)
phase_b_corr = {
    (11, 17): {
        (752, 1137): 0.7142, (4007, 794): 0.7132, (3371, 794): 0.7084,
        (769, 1137): 0.6959, (4007, 3087): 0.6952, (3298, 1137): 0.6938,
        (4007, 1109): 0.6911, (4007, 2264): 0.6886, (4007, 1676): 0.6878,
        (769, 4003): 0.6872,
    },
    (17, 23): {
        (3582, 1271): 0.6207, (3582, 694): 0.6162, (1149, 1541): 0.6076,
        (2469, 1541): 0.6007, (1137, 1271): 0.5979, (3582, 600): 0.5957,
        (2130, 1541): 0.5951, (3582, 3345): 0.5839, (3582, 3053): 0.5808,
        (1084, 2536): 0.5802,
    },
    (11, 23): {
        (752, 694): 0.6175, (752, 1271): 0.6159, (752, 600): 0.6064,
        (3791, 2536): 0.6043, (2509, 2536): 0.6003, (769, 694): 0.5996,
        (752, 3053): 0.5989, (769, 1271): 0.5965, (769, 600): 0.5961,
        (3298, 1271): 0.5934,
    },
}

results = []
print(f'{"pair":12s} {"src":>5s} {"dst":>5s} {"corr":>7s} {"ATE":>8s} {"n_fire":>7s}')
for (a, b), edges in top_edges.items():
    for src, dst in edges:
        key = (a, b, src)
        ate_val = float('nan')
        n_f = 0
        if key in ate_final:
            ate_val = float(ate_final[key][dst])
            n_f = fire_counts[key]
        corr_val = phase_b_corr[(a,b)].get((src, dst), float('nan'))
        results.append(dict(pair=f'L{a}->L{b}', src=src, dst=dst,
                             corr=corr_val, ate=ate_val, n_fire=n_f))
        print(f'L{a}->L{b:<5d}  f{src:<4d}  f{dst:<4d}  {corr_val:+.3f}  {ate_val:+.4f}  {n_f:>5d}')

# Correlation check: does ATE scale with corr?
import numpy as np
corrs = np.array([r['corr'] for r in results if not np.isnan(r['ate'])])
ates  = np.array([r['ate'] for r in results if not np.isnan(r['ate'])])
if len(corrs) > 2:
    rank_r = np.corrcoef(corrs, ates)[0,1]
    print(f'\nPearson(corr, ATE) = {rank_r:.3f}')
    print(f'mean ATE = {ates.mean():.4f}, mean |ATE| = {np.abs(ates).mean():.4f}')

with open(OUT / 'ate_summary.json', 'w') as f:
    json.dump(dict(
        results=results,
        pearson_corr_vs_ate=float(rank_r) if len(corrs)>2 else None,
        mean_ate=float(ates.mean()) if len(ates)>0 else None,
    ), f, indent=2)
print('\nSaved ate_summary.json')

## Cell 8 — Upload

In [ ]:
api = HfApi()
readme = f'''---
license: mit
base_model: Qwen/Qwen3.6-35B-A3B
tags: [mechanistic-interpretability, causal-intervention, feature-circuits]
---

# Phase C-1: Causal ATE for MoE Feature Circuit Edges

Tests whether Phase B correlations (up to 0.71) reflect *causal* edges via single-feature ablation.

## Method

For each top edge `f_i (L_A) -> f_j (L_B)`:
1. Baseline forward -> measure `pre_j` at L_B
2. Intervention: subtract `z_i * W_dec_A[i]` from L_A MoE output
3. ATE = mean(pre_j_baseline - pre_j_ablated) over tokens where `f_i` fired at baseline

## Results

See `ate_summary.json`. Headline: Pearson(corr, ATE) and mean ATE per pair.

Confirms/refutes causality of Phase B circuits.
'''
with open(OUT / 'README_phase_c.md', 'w') as f: f.write(readme)

for fp in [OUT/'ate_summary.json', OUT/'README_phase_c.md']:
    try:
        api.upload_file(path_or_fileobj=str(fp), path_in_repo=f'phase_c/{fp.name}',
                        repo_id=HF_OUT, repo_type='model',
                        commit_message='FC Phase C-1: ' + fp.name)
        print(f'OK {fp.name}')
    except Exception as e:
        print(f'FAIL {fp.name}: {e}')
print(f'\nOK https://huggingface.co/{HF_OUT}/tree/main/phase_c')